# Boston Housing Violations — Preliminary Modeling

### Setup & Data

In [2]:
import numpy as np
import matplotlib as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

df = pd.read_csv("merged_violations.csv") #reads data from csv generated by data_processing.ipynd
df.head(3)

C:\Users\adria\AppData\Local\Temp\ipykernel_53284\3966795234.py:12: DtypeWarning: Columns (0: PARCEL) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("merged_violations.csv") #reads data from csv generated by data_processing.ipynd


,case_no,source,status_dttm,status,code,description,violation_stno,violation_street,violation_city,violation_zip,...,PARCEL,POINT_X,POINT_Y,OWNER,YR_BUILT,LU,LU_DESC,BLDG_TYPE,OWN_OCC,OVERALL_COND
0,V91983,building_violations,NaN,Closed,121.2,Unsafe and Dangerous,302,Sumner,East Boston,2128.0,...,104910000.0,-71.036580,42.367679,ORELLANA-SERRANO ISRAEL,1930.0,R3,THREE-FAM DWELLING,RM - Row Middle,Y,A - Average
1,V903743,building_violations,2026-04-21 10:19:12,Open,102.8,Maintenance,125,B,South Boston,2127.0,...,601360000.0,-71.053080,42.341270,LAWRENCE PLACE CONDO TR,1899.0,CM,CONDO MAIN,LR - Low Rise,N,G - Good
2,V903738,building_violations,2026-04-21 10:07:25,Open,102.8,Maintenance,105,Third,South Boston,2127.0,...,601362000.0,-71.052757,42.341123,NOOK HILL CONDOMINIUM TRUST,2018.0,CM,CONDO MAIN,RM - Row Middle,N,A - Average


### Preprocessing
Before we can begin training a model, we need to standardize the data -- we assign frequencies to categorical features, and scale them alongside existing numerical features. Based on visualization findings, we focus on violation descriptions, neighborhood, land-use type, owner, building type, condition, coordinates, and year built.

In [3]:
cat_feat = ["description", "MAILING_NEIGHBORHOOD", "LU_DESC", "OWNER", "BLDG_TYPE", "OVERALL_COND"] #list of categorical features
for col in cat_feat:
    df[f"{col}_freq"] = df[col].map(df[col].value_counts()).fillna(0) #map frequencies to categorical features, fill in NaNs w/ 0

features = ["POINT_X", "POINT_Y", "YR_BUILT", "description_freq", "MAILING_NEIGHBORHOOD_freq", "LU_DESC_freq", "OWNER_freq", "BLDG_TYPE_freq", "OVERALL_COND_freq"] #list of all features of interest
processed_df = StandardScaler().fit_transform(df[features].fillna(0)) #standardize data by scaling features & filling missing gaps

print("Shape:", processed_df.shape)

Shape: (905678, 9)


### SVD & Fitting
We use Singular Value Decomposition to reduce dimensionality of the data, allowing us to classify the complex data stored in a building violation report.

In [4]:

def make_svd(n_comp): #following from lab, generate and fit svd
    svd = TruncatedSVD(n_components=n_comp, n_iter=15, random_state=0)
    lsa = make_pipeline(StandardScaler(), svd)
    Z = lsa.fit_transform(processed_df)  
    print("Explained variance (sum):", svd.explained_variance_ratio_.sum())
    return Z

Z = make_svd(9)

Explained variance (sum): 1.0000000000001672


### Training & Testing
We train and test the model using the holdout method of estimation, reserving a portion of the data to hopefully avoid overfitting. Because we are trying to classify and predict whether a building with mixed features has a violation, we make the assumption that all features are independent, and apply Naïve Bayes to generate predictions for the test data.

In [5]:
df["repeat_violation"] = df.groupby("PARCEL")["case_no"].transform("count") > 1 #we're looking to organize data based on parcel id, so we can identify and count repeat offenses
target = df["repeat_violation"].astype(int) #we define our target, looking to predict repeat violations

Z_train, Z_test, target_train, target_test = train_test_split(Z, target, test_size=0.25, random_state=42) #divde the data into training set & testing set

nb = GaussianNB()
nb.fit(Z_train, target_train) #create & train model
target_pred = nb.predict(Z_test) #generate predictions

prediction = pd.DataFrame({"case_no": df["case_no"].values[:len(target_pred)], "repeat_violation": target_pred}) #need to find a less convoluted way to fix indexing error
prediction.to_csv("predictions.csv", index=False)


model_score = nb.score(Z_test, target_test)
print(f"Score method - Accuracy: {model_score}")
prediction.head()



Score method - Accuracy: 0.9692871654447487


,case_no,repeat_violation
0,V91983,1
1,V903743,1
2,V903738,0
3,V903709,0
4,V903676,1
